# 3 · Transfer Learning — ResNet50

ResNet50 introduced **residual (skip) connections**, which let very deep networks train without the vanishing-gradient problem.

**Steps:** download data → load → visualise → preprocess → load pretrained base → train head → plot curves → confusion matrix.

> Tip: turn on the GPU — *Runtime → Change runtime type → GPU*.

### Step 1 — Imports

In [ ]:
# Standard imports
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("TensorFlow version:", tf.__version__)
print("GPU detected      :", tf.config.list_physical_devices("GPU"))
# If the list above is empty, enable the GPU:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU


### Step 2 — Set up dataset paths

In [ ]:
# Point to the local full dog-vs-cat dataset
# (already extracted under ./data/dog-cat-full-dataset/)
base_dir  = os.path.join("data", "dog-cat-full-dataset", "data")
train_dir = os.path.join(base_dir, "train")   # ~10 000 cats + ~10 000 dogs
val_dir   = os.path.join(base_dir, "test")     # 2 500 cats + 2 500 dogs

for split in [train_dir, val_dir]:
    for cls in sorted(os.listdir(split)):
        n = len(os.listdir(os.path.join(split, cls)))
        print(f"{split}/{cls}: {n} images")


### Step 3 — Load images into `tf.data` datasets

In [ ]:
# Load images straight from the folders. Sub-folder names become the labels.
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=123,
)

# shuffle=False on validation keeps the order fixed so the confusion
# matrix labels line up with the predictions later.
val_ds = keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)


### Step 4 — Look at the data

In [ ]:
# Peek at a few training images
plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()


### Step 5 — Apply ResNet50's own preprocessing

In [ ]:
# Each pretrained model has its OWN preprocessing. Using the wrong one
# (or plain 1/255 scaling) gives poor accuracy, so we import the matching one.
from tensorflow.keras.applications.resnet50 import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds):
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(AUTOTUNE)

train_prep = prepare(train_ds)
val_prep   = prepare(val_ds)


### Step 6 — Build the model on the ResNet50 base

In [ ]:
# Load the pretrained convolutional base (ImageNet weights, no top classifier)
base_model = keras.applications.ResNet50(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')

# Freeze it: we only train a fresh classification head on top.
# This is "feature extraction" transfer learning.
base_model.trainable = False
print("Base model layers:", len(base_model.layers), "| trainable:", base_model.trainable)

model = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
], name="resnet50_transfer")

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()


### Step 7 — Train the classification head

In [ ]:
EPOCHS = 10
history = model.fit(train_prep, validation_data=val_prep, epochs=EPOCHS)


### Step 8 — Accuracy & loss curves

In [ ]:
# Training vs validation curves — the quickest way to spot over/under-fitting
acc      = history.history["accuracy"]
val_acc  = history.history["val_accuracy"]
loss     = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     marker="o", label="Train")
plt.plot(epochs_range, val_acc, marker="o", label="Validation")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     marker="o", label="Train")
plt.plot(epochs_range, val_loss, marker="o", label="Validation")
plt.title("Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Step 9 — Confusion matrix & report

In [ ]:
# Evaluate + confusion matrix + per-class report
val_loss_v, val_acc_v = model.evaluate(val_prep, verbose=0)
print(f"Validation accuracy: {val_acc_v:.4f}")

# True labels (val_ds was NOT shuffled, so the order is stable)
y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)

# Predicted labels
y_prob = model.predict(val_prep)
y_pred = np.argmax(y_prob, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


### Step 10 — See predictions

In [ ]:
# Visualise predictions on a batch of validation images
plt.figure(figsize=(12, 12))
for images, labels in val_ds.take(1):
    probs = model.predict(preprocess_input(tf.identity(images)), verbose=0)
    preds = np.argmax(probs, axis=1)
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        true_lbl = class_names[labels[i]]
        pred_lbl = class_names[preds[i]]
        color = "green" if true_lbl == pred_lbl else "red"
        plt.title(f"pred: {pred_lbl}\ntrue: {true_lbl}", color=color, fontsize=10)
        plt.axis("off")
plt.tight_layout()
plt.show()


## Optional: fine-tuning

Feature extraction (above) trains only the new head and is usually enough for a
small dataset. To squeeze out more accuracy you can **fine-tune**: unfreeze the
top layers of the base model and train again with a *low* learning rate.

```python
base_model.trainable = True
# keep the earliest layers frozen; only fine-tune the top ~30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),   # low LR is important
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history_ft = model.fit(train_prep, validation_data=val_prep, epochs=5)
```
